# Player model: current level, weaknesses, confidence, departure from history

**Milestones:** "Estimate player habits from game history" and the fallback behaviour "when the
player model is uncertain, or the player departs sharply from their history, play solid
strength-limited moves" (README Teaching; "The player model can be wrong" row).

The model keeps one `GameSummary` per analysed game (built from `habits.MoveAnalysis`) and gives:

* **Level**: Elo estimate from ACPL with a confidence interval. The ACPL->Elo formula below is a
  *placeholder* (`TODO-calibrate`): ACPL depends heavily on the analyst engine, depth and time
  control, so it must be refit on games analysed by *our* analyst with known ratings.
* **Weaknesses**: blunder / mistake / missed-forcing-move rates with Wilson intervals, and ACPL per
  phase.
* **Confidence** in 0..1: how much data backs the estimate (placeholder formula, documented).
* **OOD score**: how far a new game is from the player's history (max |z| over features). High =
  the player is behaving unlike their past (new openings, much stronger/weaker, hiding strength).

**TODO checklist**
- [ ] Level: `estimate_elo_from_acpl`, `acpl_interval`.
- [ ] Game summaries (`summarize_game`, `GameSummary` rate properties).
- [ ] `PlayerModel`: update, confidence, weaknesses (Wilson), estimate, OOD z-score, JSON save/load.
- [ ] Self-checks (plan at the bottom).
- [ ] TODO-calibrate `estimate_elo_from_acpl` (fit on rated games analysed by our analyst).
- [ ] Recency weighting: a returning player's old games should count less (README "rust").
- [ ] Weakness relative to expected rates *at the player's level* (needs reference data).
- [ ] Answer the README open question: "from how many games?" (min_games in game.yaml).

In [ ]:
%run ./habits.ipynb
%run ./metrics.ipynb

In [ ]:
import json
import math
import statistics
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Optional

CONF_CFG = TEACH_CFG["confidence"]

## Level from ACPL (placeholder, TODO-calibrate)

Suggested starting shape: `elo = 3100 * exp(-0.01 * acpl)`, clamped to 400..3000. Exponential
decay is a commonly used rough shape (ACPL ~ 20 for top players, ~ 100+ for beginners), but the
constants are **not** fitted. A monotone decreasing function means an ACPL interval maps directly
to an Elo interval (low ACPL bound -> high Elo bound).

In [ ]:
ELO_A, ELO_K = 3100.0, 0.01        # TODO-calibrate


def estimate_elo_from_acpl(acpl: float) -> float:
    """Placeholder ACPL -> Elo mapping (see the markdown above), clamped to 400..3000."""
    # TODO: implement
    raise NotImplementedError


def acpl_interval(per_game_acpl: list, z: float = 1.96, sd_floor: float = 15.0) -> tuple[float, float, float]:
    """(mean, low, high) of ACPL across games; sd floored so 1-2 games do not look certain."""
    # Design:
    # - Normal interval on the mean of per-game ACPLs: mean +- z * sd / sqrt(n), low clipped at 0.
    # - Floor the sd (one game has no sd at all; two near-identical games would look certain).
    #   With a single game use a wider floor.
    # - No games -> (nan, 0, cpl_cap): "could be anything".
    # TODO: implement
    raise NotImplementedError

## Game summaries

In [ ]:
FEATURES = ("acpl", "blunder_rate", "mistake_rate", "missed_forcing_rate")


@dataclass
class GameSummary:
    game_id: str
    n_moves: int
    acpl: float
    counts: dict                      # classification -> count
    phase_cpl_sum: dict               # phase -> sum of CPL
    phase_moves: dict                 # phase -> number of moves
    n_forcing: int
    n_missed_forcing: int
    mean_think_time_s: Optional[float] = None

    @property
    def blunder_rate(self) -> float:
        """Blunders per move (safe for 0 moves)."""
        # TODO: implement
        raise NotImplementedError

    @property
    def mistake_rate(self) -> float:
        """Mistakes per move (safe for 0 moves)."""
        # TODO: implement
        raise NotImplementedError

    @property
    def missed_forcing_rate(self) -> float:
        """Missed forcing moves per forcing position (0 if there were none)."""
        # TODO: implement
        raise NotImplementedError

    def feature(self, name: str) -> float:
        """Value of one of FEATURES by name (used by the OOD score)."""
        # TODO: implement
        raise NotImplementedError


def summarize_game(analyses: list[MoveAnalysis], game_id: Optional[str] = None) -> GameSummary:
    """One game's MoveAnalyses -> GameSummary (counts, sums, not rates, so games can be pooled)."""
    # Design: store counts and sums (classification counts, CPL sum and moves per phase, forcing
    # counts) so the model can pool games exactly later; ValueError on an empty list.
    # TODO: implement
    raise NotImplementedError

## Estimates

In [ ]:
@dataclass
class Weakness:
    name: str
    rate: float
    low: float
    high: float
    n: int


@dataclass
class PlayerEstimate:
    elo: float
    elo_low: float
    elo_high: float
    acpl: float
    confidence: float
    n_games: int
    n_moves: int
    weaknesses: list
    acpl_by_phase: dict


class PlayerModel:
    def __init__(self, player: str, cfg: dict = CONF_CFG):
        self.player, self.cfg = player, cfg
        self.games: list[GameSummary] = []

    # -- updating ---------------------------------------------------------------
    def add_game_summary(self, s: GameSummary) -> None:
        # TODO: implement
        raise NotImplementedError

    def update(self, analyses: list[MoveAnalysis], game_id: Optional[str] = None) -> GameSummary:
        """Call after each game with the player's analysed moves; returns the new summary."""
        # TODO: implement
        raise NotImplementedError

    @property
    def n_moves(self) -> int:
        """Total moves over all games."""
        # TODO: implement
        raise NotImplementedError

    # -- confidence -------------------------------------------------------------
    def confidence(self) -> float:
        """How much data backs the estimate, 0..1 (placeholder, TODO-calibrate with ood_testing)."""
        # Design (suggested placeholder):
        # - data term = min(1, games/min_games) * min(1, moves/min_moves)   (cfg from game.yaml)
        # - width term = 1 / (1 + elo_ci_width / 800), using the Elo interval from acpl_interval
        # - confidence = data * width; 0 with no games.
        # Decisions for you: whether these shapes match how fast the estimate actually converges.
        # TODO: implement
        raise NotImplementedError

    # -- estimate ---------------------------------------------------------------
    def weaknesses(self) -> list[Weakness]:
        """Blunder, mistake and missed-forcing rates with Wilson intervals, most certain first."""
        # Design:
        # - Pool counts over all games (blunders / moves, mistakes / moves, missed / forcing).
        # - metrics.wilson_interval for each; sort by the *lower* bound so a weakness backed by
        #   lots of data ranks above a noisy one with the same rate.
        # TODO: implement
        raise NotImplementedError

    def estimate(self) -> PlayerEstimate:
        """Current level (Elo with interval), confidence, weaknesses and ACPL per phase."""
        # Design: Elo from the mean per-game ACPL; interval from acpl_interval mapped through the
        # monotone Elo formula (low ACPL -> high Elo); pooled ACPL per phase; default Elo from
        # ENGINE_CFG when there is no history.
        # TODO: implement
        raise NotImplementedError

    # -- departure from history -------------------------------------------------
    def ood_score(self, game: GameSummary | list, sd_floor: dict | None = None) -> float:
        """max |z| of the game's features vs the history. inf if fewer than 2 past games
        (no history = cannot judge = treat as out of distribution)."""
        # Design:
        # - Accept a GameSummary or a list of MoveAnalysis (summarise it first).
        # - Per feature in FEATURES: z = |value - history mean| / max(floor, history sd).
        #   The per-feature sd floors stop a very consistent history from flagging tiny changes.
        # - Max over features: one strongly unusual feature is enough to stop trusting the model.
        # Decisions for you: the floors; whether max |z| or a combined distance works better.
        # TODO: implement
        raise NotImplementedError

    def is_ood(self, game) -> bool:
        """ood_score above cfg["max_ood_score"]."""
        # TODO: implement
        raise NotImplementedError

    # -- persistence ------------------------------------------------------------
    def save(self, path: Path | str) -> None:
        """JSON with the player name and every GameSummary (dataclasses.asdict)."""
        # TODO: implement
        raise NotImplementedError

    @classmethod
    def load(cls, path: Path | str) -> "PlayerModel":
        """Inverse of save."""
        # TODO: implement
        raise NotImplementedError

## Self-check (synthetic summaries, no engine needed)

In [ ]:
import random
import tempfile

# Self-check plan (write these as asserts once the functions above work):
# - estimate_elo_from_acpl: 0 -> 3000 (clamp), huge -> 400, monotone decreasing.
# - Helper idea `_synthetic(gid, acpl, n=40, blunders=1)` that builds a GameSummary directly
#   (move_selection.ipynb's self-check can reuse it for a "confident" model).
# - Empty model: confidence 0, ood_score inf.
# - 8 similar synthetic games (ACPL 55..65): elo_low <= elo <= elo_high, confidence in (0.5, 1],
#   n_games 8, weaknesses sorted by lower bound.
# - A similar game is not OOD; a much weaker game (ACPL 200, 8 blunders) is OOD.
# - A one-game model has lower confidence than the 8-game one.
# - update() with real analyses from habits (`black` from its self-check): 3 moves, 1 blunder.
# - save/load round trip in a temp folder keeps the games and the estimate.
# Example shape of a test:
# assert estimate_elo_from_acpl(40) > estimate_elo_from_acpl(80)
print("Self-check: not written yet (see plan above)")